<a href="https://colab.research.google.com/github/Sang-jun2/my_portfolio/blob/main/Anomaly%20detection/%EC%A7%84%EB%8F%99/%EC%A7%84%EB%8F%99%EB%AA%A8%EB%8D%B8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import glob
import pandas as pd
import numpy as np
import sklearn
import joblib
from sklearn.preprocessing import MinMaxScaler
import seaborn as sns
sns.set(color_codes=True)
import matplotlib.pyplot as plt
%matplotlib inline

from numpy.random import seed
import tensorflow as tf
import logging
logging.getLogger('tensorflow').setLevel(logging.ERROR)


from keras.layers import Input, Dropout, Dense, LSTM, TimeDistributed, RepeatVector
from keras.models import Model
from keras import regularizers

import joblib

import random
random.seed(12)  # Set the random seed
import tensorflow as tf
cur_path = "C:/Users/705-4/dataset/imbalance/"
normal_file_names = glob.glob("C:/Users/705-4/dataset/normal/"+'/normal/*.csv')

imnormal_file_names_6g = glob.glob(cur_path+'/imbalance/6g/*.csv')
imnormal_file_names_10g = glob.glob(cur_path+'/imbalance/10g/*.csv')
imnormal_file_names_15g = glob.glob(cur_path+'/imbalance/15g/*.csv')
imnormal_file_names_20g = glob.glob(cur_path+'/imbalance/20g/*.csv')
imnormal_file_names_25g = glob.glob(cur_path+'/imbalance/25g/*.csv')
imnormal_file_names_30g = glob.glob(cur_path+'/imbalance/30g/*.csv')
imnormal_file_names_35g = glob.glob(cur_path+'/imbalance/35g/*.csv')

def dataReader(path_names):
    data_n = pd.DataFrame()
    for i in path_names:
        low_data = pd.read_csv(i,header=None)
        data_n = pd.concat([data_n,low_data],ignore_index=True)
    return data_n #데이터 프레임으로 읽어들어옴
data_n = dataReader(normal_file_names)
data_6g = dataReader(imnormal_file_names_6g)
data_10g = dataReader(imnormal_file_names_10g)
data_15g = dataReader(imnormal_file_names_15g)
data_20g = dataReader(imnormal_file_names_20g)
data_25g = dataReader(imnormal_file_names_25g)
data_30g = dataReader(imnormal_file_names_30g)
import pandas as pd
import numpy as np

def ref_data(data):
    # Convert the input data to a pandas DataFrame
    data = pd.DataFrame(data)
    # Check the number of columns in the data
    num_columns = data.shape[1]
    # Define the selected indices
    selected_indices = [1, 2, 3]
    # Validate the selected indices
    valid_indices = [i for i in selected_indices if i < num_columns]
    # Select the columns with the valid indices
    selected_data = data.iloc[:, valid_indices]
    # Take the absolute value of the selected data
    selected_data_abs = np.abs(selected_data)
    # Return the absolute value of the selected data
    return selected_data_abs

data_n = ref_data(data_n)
data_6g = ref_data(data_6g)
data_10g = ref_data(data_10g)
data_15g= ref_data(data_15g)
data_20g = ref_data(data_20g)
data_25g = ref_data(data_25g)
data_30g = ref_data(data_30g)
display(data_n.head(5))

import pandas as pd
from tqdm import tqdm

data = data_n.copy() #데이터를 복사해둠, 어캐처리하기전에 원본을 유지하는용

def downSampler(data, start_idx, sampling_rate, method='mean'):
    num_samples = (len(data) - start_idx) // sampling_rate
    # 모든 샘플 구간을 나누고, 구간별로 원하는 방식의 다운샘플링 적용
    data_slices = [data.iloc[start_idx + i * sampling_rate: start_idx + (i + 1) * sampling_rate] for i in range(num_samples)]

    if method == 'mean':
        downsampled_data = pd.concat([df.mean(axis=0).to_frame().T for df in tqdm(data_slices)], ignore_index=True)
    elif method == 'median':
        downsampled_data = pd.concat([df.median(axis=0).to_frame().T for df in tqdm(data_slices)], ignore_index=True)
    elif method == 'max':
        downsampled_data = pd.concat([df.max(axis=0).to_frame().T for df in tqdm(data_slices)], ignore_index=True)
    elif method == 'min':
        downsampled_data = pd.concat([df.min(axis=0).to_frame().T for df in tqdm(data_slices)], ignore_index=True)
    else:
        raise ValueError("Invalid method. Choose from 'mean', 'median', 'max', 'min'")

    return downsampled_data

# 데이터의 70%를 학습 데이터로, 15%를 검증 데이터로, 15%를 테스트 데이터로 사용
train_size = int(0.7 * len(data))
val_size = int(0.15 * len(data))
test_size = len(data) - train_size - val_size

# 시계열 순서를 유지한 데이터 분리
x_train = data[:train_size]
x_val = data[train_size:train_size + val_size]
x_test = data[train_size + val_size:]

train = x_train
test = x_test
val=x_val
print("Shape of Train Data : {}".format(train.shape))
print("Shape of Test Data : {}".format(test.shape))
print("Shape of Test Data : {}".format(val.shape))
train = downSampler(train,0,10)#시작 인덱스0부터 10까지의 평균을 계산하여 그걸 데이터 프레임에 추가함, 이를 반복
test = downSampler(test,0,10)
test = downSampler(val,0,10)

print('train data after down sampling: ',train.shape)
print('test data after down sampling: ', test.shape)
print('test data after down sampling: ', val.shape)

scaler = MinMaxScaler()
x_final=scaler.fit_transform(train)
x_test_final=scaler.transform(test)
x_val_final=scaler.transform(val)
#time 스탭을 어떻게 바꿔볼지는 51.2khz로 샘플링 된 데이터라 아마도 128,256이 적합할텐데 그렇게 되면 학습이 10일정도로 늘어나서 그냥 짧게 설정
#3d 형식이 필요없는 ae, iof,svm
# LSTM, CNN-AE는 3D 입력 형식 필요
X_train_final = x_final.reshape(x_final.shape[0], 1, x_final.shape[1])
print('Training data shape:', X_train_final.shape)

X_tests_final = x_test_final.reshape(x_test_final.shape[0], 1, x_test_final.shape[1])
print('Test data shape:', X_tests_final.shape)

X_vals_final=x_val_final.reshape(x_val_final.shape[0], 1,x_val_final.shape[1])
print('validation shape',X_vals_final.shape)

import numpy as np
from sklearn.metrics import confusion_matrix, classification_report

# 정상 데이터로 모델 예측 수행
X_pred_normal = model.predict(X_tests_final)
test_mae_loss_normal = np.mean(np.abs(X_pred_normal - X_tests_final), axis=(1,2))

# 임계값 설정: 정상 데이터의 평균 재구성 오류에 3배의 표준 편차 추가
threshold = np.mean(test_mae_loss_normal) + 3 * np.std(test_mae_loss_normal)
print(f"Anomaly Detection Threshold: {threshold}")

# 혼합 데이터셋 생성: 99% 정상 데이터, 1% 비정상 데이터
def create_mixed_dataset(normal_data, anomaly_data, normal_ratio=0.99):
    num_normal = int(len(normal_data) * normal_ratio)
    num_anomaly = len(normal_data) - num_normal  # 나머지 1%는 비정상 데이터

    normal_samples = normal_data[:num_normal]
    anomaly_samples = anomaly_data[:num_anomaly]

    # 정상과 비정상 데이터를 결합 (혼합 데이터셋)
    mixed_data = np.vstack([normal_samples, anomaly_samples])

    # 레이블 생성: 0은 정상, 1은 비정상
    mixed_labels = np.hstack([np.zeros(num_normal), np.ones(num_anomaly)])

    return mixed_data, mixed_labels

# 비정상 데이터에서 일부 샘플 가져오기
anomaly_data_downsampled = downSampler(data_6g, 0, 10).values
anomaly_data_downsampled = anomaly_data_downsampled.reshape(-1, 1, anomaly_data_downsampled.shape[1])

# 혼합 데이터셋 생성
mixed_data, mixed_labels = create_mixed_dataset(X_tests_final, anomaly_data_downsampled)

# 혼합 데이터셋에 대한 모델 예측
X_pred_mixed = model.predict(mixed_data)
test_mae_loss_mixed = np.mean(np.abs(X_pred_mixed - mixed_data), axis=(1,2))

# 평가 함수: 혼합 데이터셋 평가
def evaluate_anomaly_detection(y_true, y_pred, threshold):
    y_pred_binary = (y_pred > threshold).astype(int)
    cm = confusion_matrix(y_true, y_pred_binary)
    tn, fp, fn, tp = cm.ravel()

    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0  # False Positive Rate
    fnr = fn / (fn + tp) if (fn + tp) > 0 else 0  # False Negative Rate

    print("Confusion Matrix:")
    print(cm)
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred_binary))
    print(f"\nFalse Positive Rate: {fpr:.4f}")
    print(f"False Negative Rate: {fnr:.4f}")

# 혼합 데이터셋 평가
print("\nEvaluation on Mixed Dataset (99% Normal, 1% Anomaly):")
evaluate_anomaly_detection(mixed_labels, test_mae_loss_mixed, threshold)

# 시각화
plt.figure(figsize=(15, 8))
plt.plot(test_mae_loss_mixed, label='Mixed Data (99% Normal, 1% Anomaly)')
plt.axhline(threshold, color='red', linestyle='--', label='Anomaly Threshold')
plt.title('Anomaly Detection - Mixed Dataset (99% Normal, 1% Anomaly)')
plt.xlabel('Sample Index')
plt.ylabel('Anomaly Score')
plt.legend()
plt.show()